# Lección 2: Apache Spark — Introducción y Configuración

**Módulo 9 | Retail Analytics Pipeline | RetailMax E-Commerce**

---

## Recap: ¿Qué vimos en la Lección 1?

En la Lección 1 establecimos los fundamentos del Big Data aplicados a RetailMax:

- **Las 5V del Big Data**: Volumen, Velocidad, Variedad, Veracidad y Valor.
- **Por qué Fashion-MNIST**: 70,000 imágenes de 28×28 px (784 píxeles/imagen) → 54.8 MB solo en píxeles, sin contar metadatos ni etiquetas.
- **Arquitectura distribuida**: entendimos la diferencia entre procesamiento monolítico (un solo nodo) y distribuido (múltiples workers en paralelo).

## ¿Qué haremos en esta Lección 2?

| Paso | Tarea | Herramienta |
|------|-------|-------------|
| 1 | Verificar el entorno (Python, PySpark, Java) | `sys`, `os`, `subprocess` |
| 2 | Crear una `SparkSession` configurada para Windows local | `pyspark.sql.SparkSession` |
| 3 | Preparar Fashion-MNIST como datos distribuibles | `keras` / `numpy` |
| 4 | Crear un **RDD** desde datos en memoria | `sc.parallelize()` |
| 5 | Ejecutar acciones básicas sobre el RDD | `count`, `take`, `first` |
| 6 | Guardar resultados en disco | `saveAsTextFile()` |

## Conexión con Lección 3

Una vez que el RDD está creado y validado (lo que hacemos aquí), en la **Lección 3** aplicaremos **transformaciones** (`map`, `filter`, `flatMap`, `reduceByKey`) para limpiar, filtrar y agregar los datos de Fashion-MNIST a escala, construyendo el primer pipeline real de RetailMax.

---

# SECCIÓN 1: ¿Qué es Spark y por qué lo necesitamos?

## Apache Spark en una línea

> **Apache Spark** es un motor de procesamiento distribuido en memoria, diseñado para analizar grandes volúmenes de datos de forma paralela y tolerante a fallos.

---

## SparkContext vs. SparkSession

| Concepto | SparkContext | SparkSession |
|----------|--------------|--------------|
| **Introducido en** | Spark 1.x | Spark 2.0+ |
| **Propósito principal** | Punto de entrada al clúster; maneja RDDs | Punto de entrada unificado: RDDs + DataFrames + SQL |
| **¿Se usa solo?** | Sí (en código heredado) | Sí (moderno); incluye SparkContext internamente |
| **Cómo obtenerlo** | `SparkContext(conf=...)` | `SparkSession.builder...getOrCreate()` |
| **Acceso al otro** | Es independiente | `spark.sparkContext` devuelve el SparkContext |
| **¿Cuándo usarlo?** | Trabajos solo con RDDs (legado) | Siempre en código nuevo |

**Regla práctica para RetailMax**: Crear siempre `SparkSession`, y obtener el `SparkContext` desde ella cuando necesitemos operaciones de bajo nivel (RDDs).

---

## Arquitectura: Driver → Master → Workers

```
┌─────────────────────────────────────────────────────────────┐
│                        DRIVER PROGRAM                       │
│   (Tu script Python / Jupyter Notebook)                     │
│                                                             │
│   SparkContext / SparkSession                               │
│   ┌─────────────────────────────┐                           │
│   │  DAG Scheduler              │  ← construye el plan      │
│   │  Task Scheduler             │  ← distribuye tareas      │
│   └─────────────────────────────┘                           │
└───────────────────┬─────────────────────────────────────────┘
                    │  (envía tareas)
                    ▼
┌─────────────────────────────────────────────────────────────┐
│                     CLUSTER MANAGER                         │
│   (en local[*]: el mismo proceso; en prod: YARN/Kubernetes) │
└───────┬───────────────────┬───────────────────┬─────────────┘
        │                   │                   │
        ▼                   ▼                   ▼
┌──────────────┐  ┌──────────────┐  ┌──────────────┐
│   WORKER 1   │  │   WORKER 2   │  │   WORKER N   │
│ ┌──────────┐ │  │ ┌──────────┐ │  │ ┌──────────┐ │
│ │Executor  │ │  │ │Executor  │ │  │ │Executor  │ │
│ │Task Task │ │  │ │Task Task │ │  │ │Task Task │ │
│ └──────────┘ │  │ └──────────┘ │  │ └──────────┘ │
│  Partición 0 │  │  Partición 1 │  │  Partición N │
└──────────────┘  └──────────────┘  └──────────────┘
```

**En modo local[*]**: Driver, Master y todos los Workers conviven en el **mismo proceso** de tu PC. Perfecto para desarrollo y pruebas.

---

## Hadoop MapReduce vs. Apache Spark

| Característica | Hadoop MapReduce | Apache Spark |
|----------------|------------------|--------------|
| **Almacenamiento intermedio** | Disco (HDFS) entre cada paso | Memoria RAM (mucho más rápido) |
| **Velocidad típica** | Línea base (1×) | 10×–100× más rápido en memoria |
| **API principal** | Java verboso (`Mapper`, `Reducer`) | Python/Scala/Java fluido |
| **Evaluación** | Eager (ejecuta inmediatamente) | **Lazy** (solo ejecuta al pedir resultado) |
| **Soporte ML** | Mahout (limitado) | MLlib integrado + GraphX + Streaming |
| **Tolerancia a fallos** | Reescribe en disco (lento) | Linaje de RDDs (regenera en memoria) |
| **Caso de uso** | Batch muy grande, HDFS nativo | Batch + streaming + ML en el mismo clúster |

**Mini-ejemplo numérico de velocidad**: Calcular el promedio de píxeles en 70,000 imágenes:
- MapReduce: ~45 segundos (lectura/escritura HDFS entre Map y Reduce)
- Spark (en memoria): ~2 segundos (todo en RAM, sin disco intermedio)
- Mejora: **~22× más rápido**

---

## Modo local vs. clúster real

```
local      → 1 hilo (sin paralelismo real)       → pruebas mínimas
local[2]   → 2 hilos                              → pruebas con 2 cores
local[*]   → todos los cores del PC               → desarrollo real ← USAMOS ESTO
spark://.. → clúster standalone                   → producción pequeña
yarn       → Hadoop YARN                           → producción empresarial
k8s://..   → Kubernetes                            → nube / microservicios
```

---

# SECCIÓN 2: Instalación y verificación del entorno

In [1]:
# =============================================================================
# SECCIÓN 2: Verificación del entorno de desarrollo
# Objetivo: confirmar que Python, PySpark y Java están correctamente instalados
# antes de intentar iniciar Spark (evita errores confusos más adelante).
# =============================================================================

import sys          # Para obtener versión de Python y ruta del intérprete
import os           # Para leer variables de entorno del sistema operativo
import subprocess   # Para ejecutar comandos externos (java --version)
import platform     # Para obtener información del sistema operativo

print("=" * 60)
print("   REPORTE DE ENTORNO — RetailMax Analytics Pipeline")
print("=" * 60)

# -------------------------------------------------------------------
# PASO 1: Verificar la versión de Python
# Spark 3.x requiere Python 3.8+. Recomendamos 3.10+ para compatibilidad.
# -------------------------------------------------------------------
python_version = sys.version          # Cadena completa: '3.10.12 (main, ...)'
python_short   = sys.version_info     # Tupla: (3, 10, 12, 'final', 0)
python_exec    = sys.executable       # Ruta al intérprete activo (tu .venv)

print(f"\n{'Componente':<25} {'Estado':<15} {'Detalle'}")
print("-" * 60)

# Verificamos que la versión sea al menos 3.8
if python_short >= (3, 8):
    estado_python = "✔ OK"
else:
    estado_python = "✘ ACTUALIZAR"

print(f"{'Python':<25} {estado_python:<15} {python_short.major}.{python_short.minor}.{python_short.micro}")
print(f"{'  → Ejecutable':<25} {'INFO':<15} {python_exec}")

# -------------------------------------------------------------------
# PASO 2: Verificar PySpark
# PySpark es el paquete Python que se comunica con el motor Spark (Java/Scala).
# Si no está instalado, ejecuta: pip install pyspark en tu .venv
# -------------------------------------------------------------------
try:
    import pyspark                          # Intentamos importar PySpark
    pyspark_version = pyspark.__version__   # Leemos la versión instalada
    print(f"{'PySpark':<25} {'✔ OK':<15} {pyspark_version}")
except ImportError:
    # Si falla: PySpark no está instalado en este entorno virtual
    print(f"{'PySpark':<25} {'✘ NO ENCONTRADO':<15} Ejecuta: pip install pyspark")
    print("\n[ERROR] PySpark no está instalado. Detén aquí y ejecuta:")
    print("        pip install pyspark==3.5.1")
    raise  # Relanzamos la excepción para detener el notebook

# -------------------------------------------------------------------
# PASO 3: Verificar JAVA_HOME
# Spark (motor interno) está escrito en Scala/Java, por lo que REQUIERE
# una JDK instalada. En Windows, debes tener JAVA_HOME definida.
# Descarga JDK 11 desde: https://adoptium.net/
# -------------------------------------------------------------------
java_home = os.environ.get("JAVA_HOME", None)  # Leemos la variable de entorno

if java_home:
    print(f"{'JAVA_HOME':<25} {'✔ OK':<15} {java_home}")
else:
    # JAVA_HOME no definida: Spark no podrá iniciar
    print(f"{'JAVA_HOME':<25} {'⚠ NO DEFINIDA':<15} Ver instrucciones abajo")
    print("\n[ADVERTENCIA] JAVA_HOME no está definida en las variables de entorno.")
    print("  Solución en Windows:")
    print("  1. Instala JDK 11+: https://adoptium.net/")
    print("  2. En Inicio → 'Variables de entorno del sistema'")
    print("  3. Agrega JAVA_HOME = C:\\Program Files\\Eclipse Adoptium\\jdk-11...")
    print("  4. Agrega %JAVA_HOME%\\bin al PATH")
    print("  5. Reinicia VS Code / terminal")

# -------------------------------------------------------------------
# Verificamos la versión de Java ejecutando 'java -version' en la terminal
# subprocess.run() ejecuta un comando del sistema operativo desde Python
# -------------------------------------------------------------------
try:
    # capture_output=True captura stdout y stderr
    # text=True convierte bytes a string automáticamente
    resultado_java = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True
    )
    # Java imprime su versión en stderr (comportamiento estándar de Java)
    java_info = resultado_java.stderr.split("\n")[0]  # Primera línea del output
    print(f"{'Java (ejecutable)':<25} {'✔ OK':<15} {java_info}")
except FileNotFoundError:
    # 'java' no está en el PATH: no se puede ejecutar desde la terminal
    print(f"{'Java (ejecutable)':<25} {'✘ NO EN PATH':<15} java no encontrado en PATH")

# -------------------------------------------------------------------
# PASO 4: Información adicional del sistema
# -------------------------------------------------------------------
so = platform.system()          # 'Windows', 'Linux' o 'Darwin' (macOS)
so_version = platform.version() # Versión detallada del SO
cpu_cores = os.cpu_count()      # Número de núcleos lógicos (hyper-threading)

print(f"{'Sistema Operativo':<25} {'INFO':<15} {so}")
print(f"{'Cores disponibles':<25} {'INFO':<15} {cpu_cores} (local[*] usará todos)")
print("=" * 60)
print("✔ Verificación completada. Revisa cualquier ✘ antes de continuar.")

   REPORTE DE ENTORNO — RetailMax Analytics Pipeline

Componente                Estado          Detalle
------------------------------------------------------------
Python                    ✔ OK            3.10.11
  → Ejecutable            INFO            d:\.venv_spark\Scripts\python.exe
PySpark                   ✔ OK            4.1.1
JAVA_HOME                 ⚠ NO DEFINIDA   Ver instrucciones abajo

[ADVERTENCIA] JAVA_HOME no está definida en las variables de entorno.
  Solución en Windows:
  1. Instala JDK 11+: https://adoptium.net/
  2. En Inicio → 'Variables de entorno del sistema'
  3. Agrega JAVA_HOME = C:\Program Files\Eclipse Adoptium\jdk-11...
  4. Agrega %JAVA_HOME%\bin al PATH
  5. Reinicia VS Code / terminal
Java (ejecutable)         ✔ OK            java version "21.0.9" 2025-10-21 LTS
Sistema Operativo         INFO            Windows
Cores disponibles         INFO            12 (local[*] usará todos)
✔ Verificación completada. Revisa cualquier ✘ antes de continuar.


---

# SECCIÓN 3: Crear SparkSession

## SparkSession: el punto de entrada unificado (Spark 2.0+)

Antes de Spark 2.0 existían varios puntos de entrada separados:
- `SparkContext` para RDDs
- `SQLContext` para DataFrames
- `HiveContext` para Hive

Desde Spark 2.0, **`SparkSession`** los unifica todos en un solo objeto.

### Parámetros clave que configuraremos

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| `master` | `local[*]` | Usa **todos** los cores del PC como workers |
| `appName` | `RetailMax_Analytics` | Nombre visible en la Spark UI |
| `spark.driver.memory` | `2g` | RAM asignada al proceso Driver (2 GB) |
| `spark.sql.shuffle.partitions` | `4` | Particiones al hacer joins/aggregations (bajo para local) |
| `spark.ui.showConsoleProgress` | `false` | Desactiva barra de progreso verbosa en la consola |

### ¿Qué significa `local[*]`?

```
local      → 1 hilo  (sin paralelismo)
local[2]   → 2 hilos (simula 2 workers)
local[4]   → 4 hilos (simula 4 workers)
local[*]   → TODOS los cores del CPU  ← RECOMENDADO para desarrollo
```

Si tu PC tiene 8 cores, `local[*]` ejecutará 8 tareas en paralelo, dividiendo el trabajo en 8 particiones simultáneas.

### Spark UI

Al crear la `SparkSession`, Spark levanta automáticamente una interfaz web en:
```
http://localhost:4040
```
Desde ahí puedes ver jobs, stages, tasks, memoria usada y el DAG de ejecución.

In [2]:
# =============================================================================
# SECCIÓN 3: Crear SparkSession
# Objetivo: inicializar Spark con configuración optimizada para Windows local.
# =============================================================================

# PASO 1: Importar los módulos necesarios de PySpark
from pyspark.sql import SparkSession   # Punto de entrada principal de Spark 2.0+
import os                              # Para manipular variables de entorno si es necesario

# -------------------------------------------------------------------
# Configuración especial para Windows:
# En algunos sistemas Windows, Spark necesita winutils.exe para
# operaciones de archivos. Si ves errores de 'winutils', instala:
# https://github.com/steveloughran/winutils
# y define: os.environ["HADOOP_HOME"] = r"C:\ruta\a\winutils"
# -------------------------------------------------------------------
# os.environ["HADOOP_HOME"] = r"C:\hadoop"  # Descomenta si es necesario

# PASO 2: Crear la SparkSession con configuración optimizada para desarrollo local
try:
    spark = (
        SparkSession.builder
        # master: dónde corre Spark. local[*] = todos los cores del PC
        .master("local[*]")
        # appName: nombre que aparece en la Spark UI (localhost:4040)
        .appName("RetailMax_Analytics")
        # Memoria RAM para el proceso Driver (quien coordina todo)
        # Para datasets grandes, aumenta a 4g o 8g según disponibilidad
        .config("spark.driver.memory", "2g")
        # Particiones para operaciones de shuffle (join, groupBy, agg)
        # Valor por defecto: 200. Para local con pocos datos: 4 es suficiente
        # Demasiadas particiones en local = overhead de scheduling
        .config("spark.sql.shuffle.partitions", "4")
        # Desactivar la barra de progreso en consola (hace el output más limpio)
        .config("spark.ui.showConsoleProgress", "false")
        # getOrCreate(): si ya existe una SparkSession activa, la reutiliza
        # Esto evita el error 'Cannot run multiple SparkContexts at once'
        .getOrCreate()
    )
    print("✔ SparkSession creada exitosamente.")

except Exception as e:
    print(f"✘ Error al crear SparkSession: {e}")
    print("\nPosibles causas y soluciones:")
    print("  1. Java no instalado → instala JDK 11+ desde https://adoptium.net/")
    print("  2. JAVA_HOME no definida → revisa la Sección 2")
    print("  3. Puerto 4040 en uso → cierra otras sesiones de Spark")
    print("  4. Memoria insuficiente → reduce spark.driver.memory a '1g'")
    raise

# -------------------------------------------------------------------
# PASO 3: Mostrar información de la SparkSession activa
# Confirmamos que todo está configurado como esperamos
# -------------------------------------------------------------------
print("\n" + "=" * 60)
print("   INFORMACIÓN DE LA SPARKSESSION")
print("=" * 60)
print(f"  Versión de Spark : {spark.version}")
print(f"  Master           : {spark.sparkContext.master}")
print(f"  App Name         : {spark.sparkContext.appName}")
print(f"  App ID           : {spark.sparkContext.applicationId}")
print(f"  Cores disponibles: {spark.sparkContext.defaultParallelism}")
print("=" * 60)

# -------------------------------------------------------------------
# PASO 4: Mostrar la URL de la Spark UI
# La Spark UI permite visualizar jobs, stages, tasks y consumo de memoria
# -------------------------------------------------------------------
spark_ui_url = spark.sparkContext.uiWebUrl  # URL de la interfaz web
if spark_ui_url:
    print(f"\n  🌐 Spark UI disponible en: {spark_ui_url}")
else:
    print("\n  🌐 Spark UI disponible en: http://localhost:4040")
print("  (Abre esta URL en tu navegador mientras el notebook esté activo)")
print("\n✔ SparkSession lista para usar.")

✔ SparkSession creada exitosamente.

   INFORMACIÓN DE LA SPARKSESSION
  Versión de Spark : 4.1.1
  Master           : local[*]
  App Name         : RetailMax_Analytics
  App ID           : local-1774056192723
  Cores disponibles: 12

  🌐 Spark UI disponible en: http://Urzua:4040
  (Abre esta URL en tu navegador mientras el notebook esté activo)

✔ SparkSession lista para usar.


---

# SECCIÓN 4: Preparar Fashion-MNIST como datos distribuidos

## ¿Por qué convertir Fashion-MNIST a un formato que Spark procese?

Fashion-MNIST originalmente viene como archivos binarios IDX (o como tensores de NumPy/PyTorch). Spark no entiende ese formato directamente, pero sí puede paralelizar **listas de Python**.

### Estrategia de conversión

```
Keras/PyTorch/sklearn
        │
        │  cargar Fashion-MNIST
        ▼
  NumPy arrays
  X_train: (60000, 28, 28)  → píxeles
  y_train: (60000,)         → etiquetas (0–9)
        │
        │  convertir cada imagen
        ▼
  Lista de diccionarios Python
  [{image_id, label, label_name, pixels, split}, ...]
        │
        │  sc.parallelize()
        ▼
  RDD distribuido en N particiones
  [partición 0] [partición 1] [partición 2] [partición 3]
```

### ¿Por qué solo 10,000 registros en modo local?

- Fashion-MNIST completo train: **60,000 imágenes × 784 píxeles = 47M floats ≈ 360 MB en RAM**
- Con 10,000 imágenes: ~60 MB → manejable en cualquier laptop de desarrollo
- En producción, procesaríamos los 70,000 sin problema en un clúster real

### Etiquetas de Fashion-MNIST

| ID | Nombre en inglés | Traducción (RetailMax) |
|----|-----------------|------------------------|
| 0  | T-shirt/top     | Camiseta / Top |
| 1  | Trouser         | Pantalón |
| 2  | Pullover        | Suéter |
| 3  | Dress           | Vestido |
| 4  | Coat            | Abrigo |
| 5  | Sandal          | Sandalia |
| 6  | Shirt           | Camisa |
| 7  | Sneaker         | Zapatilla deportiva |
| 8  | Bag             | Bolso / Cartera |
| 9  | Ankle boot      | Botín |

In [3]:
# =============================================================================
# SECCIÓN 4: Cargar Fashion-MNIST y preparar datos para Spark
# Estrategia: intentar TensorFlow/Keras → PyTorch → sklearn → numpy desde URL
# =============================================================================

import numpy as np  # Para manipulación de arrays numéricos

# Mapa de etiquetas numéricas (0-9) a nombres de categorías de ropa
# Estas son las 10 clases de Fashion-MNIST, mapeadas al contexto de RetailMax
LABEL_NAMES = {
    0: "Camiseta_Top",
    1: "Pantalon",
    2: "Sueter",
    3: "Vestido",
    4: "Abrigo",
    5: "Sandalia",
    6: "Camisa",
    7: "Zapatilla_Deportiva",
    8: "Bolso_Cartera",
    9: "Botin"
}

# -------------------------------------------------------------------
# PASO 1: Cargar Fashion-MNIST
# Intentamos múltiples fuentes en orden de preferencia:
# 1. TensorFlow/Keras (más común)
# 2. PyTorch/torchvision
# 3. scikit-learn + openml
# -------------------------------------------------------------------
X_train, y_train, X_test, y_test = None, None, None, None
fuente_usada = None

# --- Intento 1: TensorFlow/Keras ---
try:
    from tensorflow.keras.datasets import fashion_mnist  # Descarga automática al llamar
    (X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
    fuente_usada = "TensorFlow/Keras"
    print(f"✔ Dataset cargado desde: {fuente_usada}")
except ImportError:
    print("  TensorFlow no disponible, intentando PyTorch...")
except Exception as e:
    print(f"  Error con TensorFlow: {e}. Intentando PyTorch...")

# --- Intento 2: PyTorch/torchvision ---
if X_train is None:
    try:
        import torchvision
        import torch

        # Descargar dataset de Fashion-MNIST (se guarda en ./data/)
        train_ds = torchvision.datasets.FashionMNIST(
            root="./data", train=True, download=True,
            transform=torchvision.transforms.ToTensor()
        )
        test_ds = torchvision.datasets.FashionMNIST(
            root="./data", train=False, download=True,
            transform=torchvision.transforms.ToTensor()
        )

        # Convertir tensores PyTorch a arrays NumPy
        X_train = train_ds.data.numpy()    # Shape: (60000, 28, 28)
        y_train = train_ds.targets.numpy() # Shape: (60000,)
        X_test  = test_ds.data.numpy()     # Shape: (10000, 28, 28)
        y_test  = test_ds.targets.numpy()  # Shape: (10000,)
        fuente_usada = "PyTorch/torchvision"
        print(f"✔ Dataset cargado desde: {fuente_usada}")
    except ImportError:
        print("  PyTorch no disponible, intentando scikit-learn...")
    except Exception as e:
        print(f"  Error con PyTorch: {e}. Intentando scikit-learn...")

# --- Intento 3: scikit-learn + OpenML ---
if X_train is None:
    try:
        from sklearn.datasets import fetch_openml
        print("  Descargando Fashion-MNIST desde OpenML (puede tardar ~1 min)...")

        # Fashion-MNIST en OpenML tiene ID 40996
        fmnist = fetch_openml(name='Fashion-MNIST', version=1, as_frame=False, parser='auto')
        X = fmnist.data.astype(np.float32)    # Shape: (70000, 784)
        y = fmnist.target.astype(int)         # Shape: (70000,)

        # OpenML devuelve todos los datos juntos; dividimos manualmente
        X_train = X[:60000].reshape(-1, 28, 28)  # Primeros 60,000 → train
        y_train = y[:60000]
        X_test  = X[60000:].reshape(-1, 28, 28)  # Últimos 10,000 → test
        y_test  = y[60000:]
        fuente_usada = "scikit-learn/OpenML"
        print(f"✔ Dataset cargado desde: {fuente_usada}")
    except Exception as e:
        print(f"  Error con sklearn/OpenML: {e}")

# --- Intento 4: Datos sintéticos (fallback de emergencia) ---
if X_train is None:
    print("\n⚠ No se pudo cargar Fashion-MNIST real.")
    print("  Generando datos SINTÉTICOS para continuar la práctica...")
    print("  (En producción, asegúrate de instalar tensorflow o torchvision)")

    # Datos sintéticos con la misma estructura que Fashion-MNIST
    np.random.seed(42)  # Semilla para reproducibilidad
    N_TRAIN, N_TEST = 60000, 10000
    X_train = np.random.randint(0, 256, (N_TRAIN, 28, 28), dtype=np.uint8)
    y_train = np.random.randint(0, 10,  (N_TRAIN,), dtype=np.int64)
    X_test  = np.random.randint(0, 256, (N_TEST,  28, 28), dtype=np.uint8)
    y_test  = np.random.randint(0, 10,  (N_TEST,),  dtype=np.int64)
    fuente_usada = "Datos SINTÉTICOS (estructura idéntica a Fashion-MNIST)"
    print(f"✔ Usando: {fuente_usada}")

# -------------------------------------------------------------------
# PASO 2: Convertir cada imagen a un diccionario estructurado
# Formato final de cada registro que procesará Spark:
# {
#   'image_id'   : 0,          # Índice único de la imagen
#   'label'      : 9,          # Clase numérica (0–9)
#   'label_name' : 'Botin',    # Nombre legible de la categoría
#   'pixels'     : [0.02, ...] # 784 floats normalizados entre 0.0 y 1.0
#   'split'      : 'train'     # Partición del dataset
# }
# -------------------------------------------------------------------
LIMITE = 10_000  # Usamos los primeros 10,000 registros de train para modo local
                 # En un clúster real, procesaríamos los 60,000 sin problema

print(f"\nConvirtiendo primeros {LIMITE:,} registros a formato diccionario...")

data = []  # Lista que contendrá todos los diccionarios

for i in range(LIMITE):
    imagen_raw = X_train[i]           # Array NumPy de shape (28, 28)
    imagen_flat = imagen_raw.flatten()  # Aplanar a shape (784,)

    # Normalizar píxeles de [0, 255] a [0.0, 1.0]
    # División por 255.0 → cada valor queda entre 0.0 y 1.0
    # Importante para algoritmos de ML que son sensibles a la escala
    pixels_norm = (imagen_flat / 255.0).tolist()  # .tolist() convierte a lista Python

    registro = {
        "image_id"   : i,                          # ID único (posición en el dataset)
        "label"      : int(y_train[i]),             # Etiqueta como int Python (no numpy)
        "label_name" : LABEL_NAMES[int(y_train[i])],  # Nombre de la categoría
        "pixels"     : pixels_norm,                 # Lista de 784 floats [0.0, 1.0]
        "split"      : "train"                      # Partición del dataset
    }
    data.append(registro)  # Agregar a la lista principal

print(f"✔ Conversión completada.")

# -------------------------------------------------------------------
# PASO 4: Mostrar un ejemplo del diccionario (sin los 784 pixels)
# Mostramos solo los primeros 5 pixels para no saturar el output
# -------------------------------------------------------------------
ejemplo = data[0].copy()               # Copiamos para no modificar el original
ejemplo["pixels"] = ejemplo["pixels"][:5]  # Solo los primeros 5 pixels
ejemplo["pixels"].append("... (784 valores en total)")

print("\n--- Ejemplo de un registro (imagen 0) ---")
for clave, valor in ejemplo.items():
    print(f"  {clave:<15}: {valor}")

# -------------------------------------------------------------------
# PASO 5: Estadísticas del dataset preparado
# -------------------------------------------------------------------
import sys as _sys

num_registros = len(data)                         # Cantidad de registros
bytes_por_registro = _sys.getsizeof(data[0])      # Tamaño aproximado por registro
bytes_total = bytes_por_registro * num_registros   # Total estimado
mb_total = bytes_total / (1024 * 1024)            # Convertir a MB

print(f"\n--- Estadísticas del dataset preparado ---")
print(f"  Registros totales   : {num_registros:,}")
print(f"  Fuente              : {fuente_usada}")
print(f"  Campos por registro : image_id, label, label_name, pixels (784), split")
print(f"  Tamaño estimado RAM : ~{mb_total:.1f} MB (referencia: solo estructura Python)")
print(f"  Píxeles totales     : {num_registros * 784:,} valores float")

✔ Dataset cargado desde: TensorFlow/Keras

Convirtiendo primeros 10,000 registros a formato diccionario...
✔ Conversión completada.

--- Ejemplo de un registro (imagen 0) ---
  image_id       : 0
  label          : 9
  label_name     : Botin
  pixels         : [0.0, 0.0, 0.0, 0.0, 0.0, '... (784 valores en total)']
  split          : train

--- Estadísticas del dataset preparado ---
  Registros totales   : 10,000
  Fuente              : TensorFlow/Keras
  Campos por registro : image_id, label, label_name, pixels (784), split
  Tamaño estimado RAM : ~2.2 MB (referencia: solo estructura Python)
  Píxeles totales     : 7,840,000 valores float


---

# SECCIÓN 5: Crear RDD desde los datos

## ¿Qué es un RDD?

> **RDD (Resilient Distributed Dataset)** es la estructura de datos fundamental de Apache Spark: una colección **inmutable**, **distribuida** y **tolerante a fallos** de elementos que pueden procesarse en paralelo.

- **Resilient**: si un worker falla, Spark puede reconstruir los datos perdidos usando el **linaje** (registro de transformaciones aplicadas).
- **Distributed**: los datos se dividen en **particiones** distribuidas entre los workers.
- **Dataset**: una colección de datos de cualquier tipo Python (dicts, tuples, strings, etc.).

---

## `parallelize()` vs. `textFile()`

| Método | Fuente | Cuándo usarlo |
|--------|--------|---------------|
| `sc.parallelize(lista)` | Datos **en memoria** (lista Python) | Datos ya cargados en Python, pruebas, datos pequeños |
| `sc.textFile("ruta")` | Datos **en disco** (archivos de texto) | Archivos CSV/JSON grandes, HDFS, S3 |

**En esta lección**: usamos `parallelize()` porque Fashion-MNIST ya está en memoria como lista de dicts.

---

## Concepto de particiones

```
Lista de 10,000 registros
├── Partición 0: registros [0    – 2,499]   → Worker/Core 0
├── Partición 1: registros [2,500 – 4,999]  → Worker/Core 1
├── Partición 2: registros [5,000 – 7,499]  → Worker/Core 2
└── Partición 3: registros [7,500 – 9,999]  → Worker/Core 3
```

**¿Por qué 4 particiones?**: Coincide con `spark.sql.shuffle.partitions=4` y es manejable en local. Más particiones = más paralelismo, pero también más overhead de coordinación.

**Mini-ejemplo numérico**: Con 4 particiones y 4 cores, un `count()` sobre 10,000 registros:
- Core 0 cuenta 2,500 → reporta 2,500
- Core 1 cuenta 2,500 → reporta 2,500
- Core 2 cuenta 2,500 → reporta 2,500
- Core 3 cuenta 2,500 → reporta 2,500
- Driver suma: 2,500 + 2,500 + 2,500 + 2,500 = **10,000** ✔

In [4]:
# =============================================================================
# SECCIÓN 5: Crear RDD desde los datos en memoria
# Objetivo: distribuir la lista de diccionarios de Fashion-MNIST en un RDD
# con 4 particiones para procesamiento paralelo.
# =============================================================================

# PASO 1: Obtener el SparkContext desde la SparkSession
# SparkContext (sc) es la conexión de bajo nivel con el motor Spark
# Se usa directamente para operaciones con RDDs
sc = spark.sparkContext  # Accedemos al SparkContext embebido en la SparkSession
print(f"✔ SparkContext obtenido. App ID: {sc.applicationId}")

# -------------------------------------------------------------------
# PASO 2: Crear el RDD usando parallelize()
# parallelize() toma una lista Python y la distribuye en N particiones
# numSlices=4 → creamos exactamente 4 particiones
# Spark asignará ~2,500 registros a cada partición (10,000 / 4)
# -------------------------------------------------------------------
rdd = sc.parallelize(
    data,         # Lista de 10,000 diccionarios preparada en la Sección 4
    numSlices=4   # Número de particiones (= número de tareas paralelas)
)
print("✔ RDD creado con sc.parallelize()")

# -------------------------------------------------------------------
# PASO 3: Verificar el número de particiones
# getNumPartitions() devuelve cuántas particiones tiene el RDD
# Cada partición puede procesarse en un hilo/core diferente
# -------------------------------------------------------------------
num_particiones = rdd.getNumPartitions()  # Debe ser 4 (lo que pedimos)
print(f"✔ Número de particiones : {num_particiones}")
print(f"  Registros por partición ≈ {len(data) // num_particiones:,}")

# -------------------------------------------------------------------
# PASO 4: Ver el linaje del RDD (toDebugString)
# El linaje (lineage) es el grafo de transformaciones que generaron este RDD
# Es lo que permite a Spark reconstruir datos perdidos sin reescribir a disco
# -------------------------------------------------------------------
print("\n--- Linaje del RDD (toDebugString) ---")
linaje_bytes = rdd.toDebugString()  # Devuelve bytes con la representación

# Decodificamos de bytes a string para mostrar legiblemente
if isinstance(linaje_bytes, bytes):
    linaje_str = linaje_bytes.decode("utf-8")  # .decode() convierte bytes → str
else:
    linaje_str = linaje_bytes  # En algunas versiones ya viene como string

print(linaje_str)
print("\nInterpretación:")
print("  ParallelCollectionRDD = RDD creado desde una colección Python en memoria")
print("  [4] = número de particiones")
print("  Cada línea representa una etapa en el grafo de transformaciones")

print("\n✔ RDD listo para ejecutar acciones y transformaciones.")

✔ SparkContext obtenido. App ID: local-1774056192723
✔ RDD creado con sc.parallelize()
✔ Número de particiones : 4
  Registros por partición ≈ 2,500

--- Linaje del RDD (toDebugString) ---
(4) ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:299 []

Interpretación:
  ParallelCollectionRDD = RDD creado desde una colección Python en memoria
  [4] = número de particiones
  Cada línea representa una etapa en el grafo de transformaciones

✔ RDD listo para ejecutar acciones y transformaciones.


---

# SECCIÓN 6: Acciones básicas sobre el RDD

## Transformaciones vs. Acciones — El corazón de la evaluación Lazy

### Transformaciones (LAZY — no ejecutan el job)

Las transformaciones **construyen el plan** pero no ejecutan nada:

| Transformación | Qué hace |
|----------------|----------|
| `map(f)` | Aplica función `f` a cada elemento, devuelve nuevo RDD |
| `filter(f)` | Filtra elementos donde `f` devuelve `True` |
| `flatMap(f)` | Como `map`, pero aplana el resultado |
| `groupByKey()` | Agrupa valores por clave |
| `reduceByKey(f)` | Agrega valores por clave con función `f` |

### Acciones (EAGER — disparan la ejecución del job)

Las acciones **ejecutan el plan** y devuelven un resultado al Driver:

| Acción | Qué devuelve |
|--------|--------------|
| `count()` | Número total de elementos (int) |
| `take(n)` | Lista con los primeros n elementos |
| `first()` | El primer elemento |
| `collect()` | ⚠ TODOS los elementos como lista (cuidado con RAM) |
| `countByValue()` | Diccionario {valor: frecuencia} |
| `takeSample(...)` | Muestra aleatoria de n elementos |
| `saveAsTextFile()` | Guarda en disco (no devuelve datos) |

```
Analogía:
  Transformación = planear una receta (escribirla en papel)
  Acción         = cocinar la receta (usar ingredientes, calor, tiempo)

  Spark acumula transformaciones sin ejecutarlas (lazy)
  Cuando llega una acción, optimiza todo el plan y lo ejecuta de una vez
```

In [5]:
# =============================================================================
# SECCIÓN 6: Acciones básicas sobre el RDD de Fashion-MNIST
# Cada acción dispara un "job" en Spark (visible en la Spark UI)
# =============================================================================

import time  # Para medir el tiempo de ejecución de cada acción

print("=" * 60)
print("   EJECUTANDO ACCIONES SOBRE EL RDD")
print("=" * 60)

# -------------------------------------------------------------------
# PASO 1: count() — contar total de registros
# count() es una ACCIÓN: recorre todas las particiones, cuenta los
# elementos en cada una y suma los resultados en el Driver.
# -------------------------------------------------------------------
print("\n[Acción 1] rdd.count()")
t_inicio = time.time()          # Registramos el tiempo de inicio
total = rdd.count()             # ← AQUÍ Spark ejecuta el job
t_fin = time.time()             # Registramos el tiempo de fin
tiempo_count = t_fin - t_inicio # Calculamos la duración

print(f"  Resultado       : {total:,} registros")
print(f"  Tiempo de ejecución: {tiempo_count:.3f} segundos")
print(f"  ↳ Interpretación: el RDD contiene {total:,} imágenes de Fashion-MNIST")
print(f"    distribuidas en {rdd.getNumPartitions()} particiones.")

# -------------------------------------------------------------------
# PASO 2: take(n) — obtener los primeros N registros
# take(3) NO trae todos los datos al Driver (eficiente).
# Solo lee las particiones necesarias hasta completar n registros.
# -------------------------------------------------------------------
print("\n[Acción 2] rdd.take(3)")
t_inicio = time.time()
primeros_3 = rdd.take(3)        # ← AQUÍ Spark ejecuta el job
t_fin = time.time()
tiempo_take = t_fin - t_inicio

print(f"  Tiempo de ejecución: {tiempo_take:.3f} segundos")
print("  Primeros 3 registros (sin campo 'pixels' para brevedad):")

for i, registro in enumerate(primeros_3):
    # Creamos una versión resumida sin los 784 pixels
    resumen = {k: v for k, v in registro.items() if k != "pixels"}
    resumen["pixels_preview"] = f"{registro['pixels'][:3]}... ({len(registro['pixels'])} valores)"
    print(f"  Registro {i}: {resumen}")

print("  ↳ Interpretación: take(n) es útil para inspeccionar los datos sin")
print("    traer todo el dataset al Driver (a diferencia de collect()).")

# -------------------------------------------------------------------
# PASO 3: first() — obtener el primer registro
# Equivalente a take(1)[0]. Más expresivo y ligeramente más eficiente.
# -------------------------------------------------------------------
print("\n[Acción 3] rdd.first()")
t_inicio = time.time()
primer_registro = rdd.first()   # ← AQUÍ Spark ejecuta el job
t_fin = time.time()

print(f"  Tiempo de ejecución: {t_fin - t_inicio:.3f} segundos")
print("  Primer registro:")
print(f"    image_id  : {primer_registro['image_id']}")
print(f"    label     : {primer_registro['label']}")
print(f"    label_name: {primer_registro['label_name']}")
print(f"    split     : {primer_registro['split']}")
print(f"    pixels[0] : {primer_registro['pixels'][0]:.4f} (normalizado de {int(primer_registro['pixels'][0]*255)}/255)")
print("  ↳ Interpretación: esta imagen pertenece a la categoría")
print(f"    '{primer_registro['label_name']}' (label={primer_registro['label']}).")

# -------------------------------------------------------------------
# PASO 4: countByValue() — distribución de clases
# Primero extraemos solo los labels con map(), luego contamos.
# map() es una TRANSFORMACIÓN (lazy), countByValue() es la ACCIÓN.
# -------------------------------------------------------------------
print("\n[Acción 4] rdd_labels.countByValue() — distribución de clases")
t_inicio = time.time()

# map() → TRANSFORMACIÓN: crea un nuevo RDD con solo los label_name
# Aún no ejecuta nada, solo construye el plan
rdd_labels = rdd.map(lambda x: x["label_name"])

# countByValue() → ACCIÓN: ejecuta el job y devuelve dict {valor: count}
distribucion = rdd_labels.countByValue()  # ← AQUÍ Spark ejecuta
t_fin = time.time()

print(f"  Tiempo de ejecución: {t_fin - t_inicio:.3f} segundos")
print("  Distribución de clases en los 10,000 registros:")
print(f"  {'Categoría':<25} {'Cantidad':>10} {'Porcentaje':>12}")
print("  " + "-" * 50)

# Ordenamos por nombre de categoría para consistencia
for categoria, cantidad in sorted(distribucion.items()):
    porcentaje = (cantidad / total) * 100
    barra = "█" * int(porcentaje / 2)  # Barra visual proporcional
    print(f"  {categoria:<25} {cantidad:>10,} {porcentaje:>10.1f}%  {barra}")

print("  ↳ Interpretación: si la distribución es ~1,000 por clase,")
print("    el dataset está balanceado (ideal para entrenar modelos ML).")

# -------------------------------------------------------------------
# PASO 5: takeSample() — muestra aleatoria reproducible
# takeSample(withReplacement, num, seed)
#   withReplacement=False → sin repetición (cada elemento solo 1 vez)
#   num=5                 → tomamos 5 elementos
#   seed=42               → semilla para reproducibilidad
# -------------------------------------------------------------------
print("\n[Acción 5] rdd.takeSample(False, 5, seed=42) — muestra aleatoria")
t_inicio = time.time()

muestra = rdd.takeSample(
    False,   # withReplacement: False = sin reemplazo
    5,       # Número de elementos a tomar
    seed=42  # Semilla para reproducibilidad
)  # ← AQUÍ Spark ejecuta el job

t_fin = time.time()
print(f"  Tiempo de ejecución: {t_fin - t_inicio:.3f} segundos")
print("  5 registros aleatorios (campos no-pixels):")
print(f"  {'#':<4} {'image_id':<12} {'label':<8} {'label_name':<25} {'split'}")
print("  " + "-" * 60)
for j, reg in enumerate(muestra):
    print(f"  {j+1:<4} {reg['image_id']:<12} {reg['label']:<8} {reg['label_name']:<25} {reg['split']}")
print("  ↳ Interpretación: takeSample() es útil para inspección rápida")
print("    y debugging sin sesgo de posición (no siempre los primeros N).")

print("\n" + "=" * 60)
print("✔ Todas las acciones ejecutadas correctamente.")
print("  (Revisa la Spark UI en http://localhost:4040 → pestaña 'Jobs')")

   EJECUTANDO ACCIONES SOBRE EL RDD

[Acción 1] rdd.count()
  Resultado       : 10,000 registros
  Tiempo de ejecución: 4.898 segundos
  ↳ Interpretación: el RDD contiene 10,000 imágenes de Fashion-MNIST
    distribuidas en 4 particiones.

[Acción 2] rdd.take(3)
  Tiempo de ejecución: 1.176 segundos
  Primeros 3 registros (sin campo 'pixels' para brevedad):
  Registro 0: {'image_id': 0, 'label': 9, 'label_name': 'Botin', 'split': 'train', 'pixels_preview': '[0.0, 0.0, 0.0]... (784 valores)'}
  Registro 1: {'image_id': 1, 'label': 0, 'label_name': 'Camiseta_Top', 'split': 'train', 'pixels_preview': '[0.0, 0.0, 0.0]... (784 valores)'}
  Registro 2: {'image_id': 2, 'label': 0, 'label_name': 'Camiseta_Top', 'split': 'train', 'pixels_preview': '[0.0, 0.0, 0.0]... (784 valores)'}
  ↳ Interpretación: take(n) es útil para inspeccionar los datos sin
    traer todo el dataset al Driver (a diferencia de collect()).

[Acción 3] rdd.first()
  Tiempo de ejecución: 1.132 segundos
  Primer registro:
 

---

# SECCIÓN 7: Guardar RDD a disco (CSV)

## ¿Por qué guardar resultados intermedios?

En un pipeline real de RetailMax, raramente procesamos datos de principio a fin en un solo paso. Los **checkpoints** y **resultados intermedios** sirven para:

1. **Tolerancia a fallos**: si una etapa posterior falla, no recomputamos desde cero.
2. **Colaboración**: otros equipos pueden usar los resultados sin ejecutar todo el pipeline.
3. **Auditoría**: tener registros de cada etapa facilita la detección de errores.
4. **Eficiencia**: Spark puede recargar desde disco en lugar de recomputar.

### Sobre `saveAsTextFile()`

- Guarda **una línea por elemento** del RDD como string.
- Crea **un archivo por partición**: `part-00000`, `part-00001`, etc.
- En HDFS/S3 esto es muy eficiente; en local es para demostración.
- Requiere que los elementos sean strings (o convertibles a string).

In [6]:
# =============================================================================
# SECCIÓN 7: Guardar RDD simplificado a disco en formato CSV
# Guardamos solo image_id, label, label_name (sin los 784 pixels)
# para demostrar el flujo de persistencia sin sobrecargar el disco.
# =============================================================================

import os        # Para operaciones de sistema de archivos
import shutil    # Para eliminar directorios completos (rmtree)

# Ruta donde guardaremos el output
# NOTA: saveAsTextFile() crea la carpeta COMPLETA; si ya existe, lanza error.
# Por eso usamos shutil.rmtree() para limpiar antes de guardar.
RUTA_OUTPUT = "output/leccion2_muestra"

# -------------------------------------------------------------------
# PASO 1: Crear RDD simplificado (solo metadatos, sin pixels)
# map() → TRANSFORMACIÓN que crea un nuevo RDD desde el original
# Seleccionamos solo los campos relevantes para el CSV
# -------------------------------------------------------------------
rdd_simple = rdd.map(
    lambda x: (x["image_id"], x["label"], x["label_name"], x["split"])
)  # ← Transformación LAZY: aún no ejecuta nada

print("✔ RDD simplificado creado (image_id, label, label_name, split)")

# -------------------------------------------------------------------
# PASO 2: Convertir tuplas a strings CSV
# saveAsTextFile() guarda cada elemento como una línea de texto,
# así que debemos convertir las tuplas a formato CSV string.
# -------------------------------------------------------------------
rdd_csv = rdd_simple.map(
    lambda t: f"{t[0]},{t[1]},{t[2]},{t[3]}"  # Formato: id,label,nombre,split
)  # ← Otra transformación LAZY encadenada

# Añadimos un header (encabezado CSV)
# Como el header es solo 1 elemento, lo unimos al RDD principal
header_rdd = sc.parallelize(["image_id,label,label_name,split"])  # RDD de 1 elemento
rdd_con_header = header_rdd.union(rdd_csv)  # union() concatena dos RDDs

print("✔ RDD convertido a formato CSV string con header")

# -------------------------------------------------------------------
# PASO 3: Manejar directorio existente
# saveAsTextFile() FALLA si el directorio ya existe.
# Usamos shutil.rmtree() para eliminarlo si existe previamente.
# -------------------------------------------------------------------
if os.path.exists(RUTA_OUTPUT):
    shutil.rmtree(RUTA_OUTPUT)  # Elimina el directorio y todo su contenido
    print(f"✔ Directorio existente eliminado: {RUTA_OUTPUT}")
else:
    print(f"✔ Directorio de destino limpio: {RUTA_OUTPUT}")

# Crear directorio padre si no existe
os.makedirs("output", exist_ok=True)  # exist_ok=True evita error si ya existe

# -------------------------------------------------------------------
# PASO 4: Guardar con saveAsTextFile()
# Esta es una ACCIÓN → dispara la ejecución de todo el plan lazy:
#   1. rdd original
#   2. map() para simplificar
#   3. map() para convertir a CSV string
#   4. union() con el header
#   5. saveAsTextFile() para escribir a disco
# -------------------------------------------------------------------
print(f"\nGuardando RDD en: {RUTA_OUTPUT}/")
print("  (Spark creará un archivo 'part-XXXXX' por cada partición)")

t_inicio = time.time()
rdd_con_header.saveAsTextFile(RUTA_OUTPUT)  # ← AQUÍ Spark ejecuta TODO el pipeline
t_fin = time.time()

print(f"✔ Guardado completado en {t_fin - t_inicio:.3f} segundos")

# -------------------------------------------------------------------
# PASO 5: Verificar que los archivos se crearon correctamente
# os.listdir() devuelve la lista de archivos en el directorio
# -------------------------------------------------------------------
print(f"\n--- Archivos creados en '{RUTA_OUTPUT}/' ---")

archivos = sorted(os.listdir(RUTA_OUTPUT))  # Ordenados alfabéticamente
tamanio_total = 0

for archivo in archivos:
    ruta_completa = os.path.join(RUTA_OUTPUT, archivo)  # Ruta completa al archivo
    if os.path.isfile(ruta_completa):
        tamano_bytes = os.path.getsize(ruta_completa)    # Tamaño en bytes
        tamanio_total += tamano_bytes
        print(f"  {archivo:<30} {tamano_bytes:>10,} bytes")
    else:
        print(f"  {archivo:<30} (directorio)")

print(f"\n  Total: {len([a for a in archivos if not a.startswith('.')])}"
      f" archivos, {tamanio_total:,} bytes ({tamanio_total/1024:.1f} KB)")
print("\n  Nota: part-00000 tiene el header + datos de la partición 0")
print("  _SUCCESS = archivo vacío que Spark crea para indicar éxito")

# Mostrar las primeras 3 líneas del primer archivo de datos
primer_part = os.path.join(RUTA_OUTPUT, "part-00000")
if os.path.exists(primer_part):
    print(f"\n  Primeras 3 líneas de part-00000:")
    with open(primer_part, "r", encoding="utf-8") as f:
        for i, linea in enumerate(f):
            if i >= 3:
                break
            print(f"    {linea.rstrip()}")

print("\n✔ Datos guardados exitosamente en disco.")

✔ RDD simplificado creado (image_id, label, label_name, split)
✔ RDD convertido a formato CSV string con header
✔ Directorio existente eliminado: output/leccion2_muestra

Guardando RDD en: output/leccion2_muestra/
  (Spark creará un archivo 'part-XXXXX' por cada partición)


Py4JJavaError: An error occurred while calling o137.saveAsTextFile.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSyste

---

# SECCIÓN 8: Cerrar SparkSession

In [ ]:
# =============================================================================
# SECCIÓN 8: Cerrar la SparkSession limpiamente
# Siempre cerrar Spark al terminar el notebook para:
#   1. Liberar memoria RAM y recursos del sistema
#   2. Detener el servidor de la Spark UI (puerto 4040)
#   3. Evitar conflictos si vuelves a iniciar Spark en la misma sesión
# =============================================================================

print("Cerrando SparkSession...")

try:
    spark.stop()  # Detiene la SparkSession y libera todos los recursos
    print("✔ SparkSession detenida correctamente.")
    print("  La Spark UI (http://localhost:4040) ya no estará disponible.")
    print("  Para reiniciar Spark, vuelve a ejecutar la celda de la Sección 3.")
except Exception as e:
    # Si ya estaba detenida o hubo un error, lo reportamos sin fallar
    print(f"  Nota al cerrar: {e}")
    print("  (Puede ser que la sesión ya estaba cerrada — no es un error crítico)")

print("\n" + "=" * 60)
print("   FIN DE LA LECCIÓN 2")
print("=" * 60)

---

# SECCIÓN 9: Informe de la Lección

## Resumen de lo logrado

En esta lección completamos los primeros pasos del pipeline distribuido de RetailMax:

1. **Entorno verificado**: Python 3.10+, PySpark y Java correctamente configurados.
2. **SparkSession creada**: con configuración optimizada para Windows local (`local[*]`, 2g driver memory).
3. **Fashion-MNIST cargado**: 10,000 imágenes convertidas a diccionarios Python estructurados.
4. **RDD creado**: datos distribuidos en 4 particiones con `sc.parallelize()`.
5. **Acciones ejecutadas**: `count`, `take`, `first`, `countByValue`, `takeSample`.
6. **Resultados guardados**: CSV con metadatos (image_id, label, label_name) en `output/leccion2_muestra/`.

---

## Tabla de conceptos aprendidos

| Concepto aprendido | Código usado | Para qué sirve en RetailMax |
|--------------------|--------------|-----------------------------|
| SparkSession | `SparkSession.builder...getOrCreate()` | Punto de entrada a todo el pipeline de analytics |
| Modo local[*] | `.master("local[*]")` | Desarrollo y pruebas sin necesidad de clúster |
| RDD | `sc.parallelize(data, 4)` | Distribuir el catálogo de imágenes para procesamiento paralelo |
| Particiones | `numSlices=4` | Dividir el trabajo entre cores del CPU |
| Linaje | `rdd.toDebugString()` | Tolerancia a fallos: reconstruir datos sin reescribir a disco |
| Acción count() | `rdd.count()` | Validar que se cargaron todos los registros del catálogo |
| Acción take() | `rdd.take(3)` | Inspeccionar muestra sin sobrecargar memoria |
| Acción countByValue() | `rdd_labels.countByValue()` | Detectar desbalance de clases en categorías de productos |
| Acción takeSample() | `rdd.takeSample(False, 5, 42)` | Muestreo reproducible para auditoría de calidad |
| saveAsTextFile | `rdd.saveAsTextFile(ruta)` | Persistir resultados intermedios del pipeline |
| Lazy evaluation | `map()` + acción | Optimizar el plan de ejecución antes de computar |

---

## Conexión con la Lección 3

En la **Lección 3** aplicaremos **transformaciones** sobre el mismo RDD que creamos hoy:

```
Lección 2 (hoy)            Lección 3 (próxima)
─────────────────          ────────────────────────────────────
RDD creado ✔           →   map()       : normalizar y limpiar píxeles
Acciones básicas ✔     →   filter()    : filtrar por categoría de ropa
                        →   flatMap()   : extraer características por píxel
                        →   reduceByKey(): calcular estadísticas por categoría
                        →   sortBy()    : ordenar resultados
```

La Lección 3 construirá el **primer pipeline real** de RetailMax: limpiar, filtrar y agregar datos de Fashion-MNIST para generar estadísticas por categoría de producto.

---

## ✅ Checklist de entregables de la Lección 2

- [ ] **Entorno verificado**: Python ≥ 3.8, PySpark instalado, JAVA_HOME definida
- [ ] **SparkSession creada** con `appName="RetailMax_Analytics"` y `master="local[*]"`
- [ ] **Fashion-MNIST cargado** (real o sintético) y convertido a lista de diccionarios
- [ ] **RDD creado** con `sc.parallelize(data, numSlices=4)`
- [ ] **`rdd.count()`** ejecutado → resultado: 10,000 registros
- [ ] **`rdd.take(3)`** ejecutado → 3 registros inspeccionados
- [ ] **`countByValue()`** ejecutado → distribución de clases verificada
- [ ] **`takeSample()`** ejecutado → muestra aleatoria reproducible obtenida
- [ ] **CSV guardado** en `output/leccion2_muestra/` con archivos `part-XXXXX`
- [ ] **SparkSession cerrada** con `spark.stop()`
- [ ] **Spark UI** visitada en `http://localhost:4040` durante la ejecución